# kaiming-uniform-init — worked example 1: Kaiming-uniform Linear with forward pass shape and bound verification

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kaiming-uniform-init`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import math
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

PyTorch's default weight initialization for `nn.Linear` uses a uniform distribution bounded by `±1/sqrt(fan_in)`. Here `fan_in` is the number of input features. Both the weight and bias are initialized with this scheme. The resulting distribution is symmetric around zero and its magnitude is controlled by the width of the previous layer.

## Worked solution

Step 1: Compute `bound = 1.0 / math.sqrt(in_features)`. For `in_features=16`, `bound ≈ 0.25`.

Step 2: Build the weight by sampling `t.empty(out_features, in_features).uniform_(-bound, bound)` into a plain tensor first, then wrap in `nn.Parameter`. Do NOT call `uniform_` on a Parameter directly — leaf tensors with `requires_grad=True` reject in-place ops.

Step 3: Repeat the same pattern for the bias (shape `(out_features,)`).

Step 4: Define `forward(self, x)` as `x @ self.weight.T + self.bias`. The transpose aligns the `in_features` axis for the dot product.

Step 5: Run a forward pass with a small input batch and print the output shape and the empirical max-abs of the weight to confirm it is `<= bound`.

In [ ]:
import torch as t
import torch.nn as nn
import math

t.manual_seed(7)

class KaimingLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        bound = 1.0 / math.sqrt(in_features)
        w_init = t.empty(out_features, in_features).uniform_(-bound, bound)
        self.weight = nn.Parameter(w_init)
        b_init = t.empty(out_features).uniform_(-bound, bound)
        self.bias = nn.Parameter(b_init)

    def forward(self, x: t.Tensor) -> t.Tensor:
        return x @ self.weight.T + self.bias

def make_kaiming_linear(in_f: int, out_f: int) -> KaimingLinear:
    return KaimingLinear(in_f, out_f)

# Exercise: in_features=16, out_features=8
layer = make_kaiming_linear(16, 8)
bound = 1.0 / math.sqrt(16)  # 0.25

print(f'bound: {bound:.4f}')
print(f'weight max-abs: {layer.weight.abs().max().item():.4f}')  # <= 0.25
print(f'bias   max-abs: {layer.bias.abs().max().item():.4f}')    # <= 0.25

# Forward pass: batch of 5
xb = t.randn(5, 16)
out = layer(xb)
print(f'output shape: {out.shape}')   # (5, 8)
print(f'output sample: {out[0].detach().numpy().round(3)}')